# 02 - Fundamentos Modernos de PLN y LLMs
## Continuación de la Clase del curso de  Inteligencia Artifial 2, desde la Sección 6: TF-IDF, Machine Learning, Embeddings y Transformers

# Descripción del Notebook

Este notebook continúa el trabajo realizado en la primera clase. En la clase anterior se trabajó hasta la sección 5, donde se introdujo NLP tradicional mediante Bag of Words. Se realizó un repaso de Jupyter notebooks en general y además se repaso el uso de la biblioteca de pandas.

**Nota:** Este notebook está diseñado para ejecutarse Google Colab.

# Objetivos de Aprendizaje

Al finalizar esta clase el estudiante podrá:

- Explicar qué es TF-IDF y ¿para qué se utiliza?
- Convertir texto en representaciones numéricas usando TF-IDF..
- Analizar palabras importantes dentro de documentos.
- Cargar y analizar un libro en formato .txt.
- Calcular similitud entre fragmentos de texto.
- Construir un clasificador básico de texto.
- Comparar TF-IDF con embeddings modernos.
- Utilizar modelos preentrenados mediante HuggingFace.
- Identificar limitaciones de NLP clásico y moderno.

# ============================================================
# Preparación del Entorno
# ============================================================

Antes de iniciar con TF-IDF y modelos modernos, se instalarán e importarán las librerías necesarias.

Este notebook está pensado para Google Colab. Algunas librerías pueden tardar un poco en instalarse la primera vez.

In [ ]:
# ============================================================
# INSTALACIÓN DE LIBRERÍAS
# ============================================================

# transformers
# Biblioteca de HuggingFace utilizada para trabajar con modelos transformer.
# Permite usar pipelines preentrenados para tareas como:
# análisis de sentimientos, resumen, question answering y generación de texto.

!pip -q install transformers


# sentence-transformers
# Librería especializada en generar embeddings semánticos.
# Permite convertir frases, párrafos y documentos en vectores numéricos
# que representan significado.

!pip -q install sentence-transformers


# scikit-learn
# Biblioteca clásica de Machine Learning.
# Incluye herramientas para TF-IDF, clasificación, división de datos,
# métricas, modelos tradicionales y similitud coseno.

!pip -q install scikit-learn


# datasets
# Librería de HuggingFace para cargar datasets.
# En este notebook queda instalada para futuras prácticas con datasets reales.

!pip -q install datasets


# accelerate
# Librería de HuggingFace que ayuda a ejecutar modelos de manera más eficiente.
# Es útil cuando se trabaja con GPU o modelos transformer más pesados.

!pip -q install accelerate


# umap-learn
# Librería para reducción de dimensionalidad.
# Se puede usar para visualizar embeddings en 2D o 3D.

!pip -q install umap-learn

# Librerías Utilizadas

En esta sección se importan las librerías que se utilizarán durante la clase.

Se utilizarán librerías para:

- Manipulación de datos.
- Visualización, representación numérica de texto.
- Modelos tradicionales de Machine Learning.
- Métricas de evaluación.
- Embeddings semánticos.
- Modelos transformer preentrenados.

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

# NumPy se utiliza para operaciones numéricas y manejo de arreglos.
import numpy as np

# Pandas se utiliza para trabajar con tablas y DataFrames.
import pandas as pd

# Matplotlib se utiliza para crear gráficos básicos.
import matplotlib.pyplot as plt

# Seaborn se utiliza para crear visualizaciones estadísticas más claras.
import seaborn as sns

# TfidfVectorizer permite convertir texto en representaciones TF-IDF.
from sklearn.feature_extraction.text import TfidfVectorizer

# CountVectorizer se incluye para comparar TF-IDF contra Bag of Words.
from sklearn.feature_extraction.text import CountVectorizer

# train_test_split permite dividir datos en entrenamiento y prueba.
from sklearn.model_selection import train_test_split

# MultinomialNB es un algoritmo clásico utilizado para clasificación de texto.
from sklearn.naive_bayes import MultinomialNB

# LogisticRegression es otro algoritmo clásico para clasificación.
from sklearn.linear_model import LogisticRegression

# Métricas para evaluar modelos.
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# cosine_similarity permite comparar vectores y medir qué tan similares son.
from sklearn.metrics.pairwise import cosine_similarity

# SentenceTransformer permite generar embeddings semánticos.
from sentence_transformers import SentenceTransformer

# pipeline permite usar modelos transformer preentrenados de HuggingFace.
from transformers import pipeline

# Warnings permite controlar advertencias para mantener el notebook más limpio.
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# VERIFICAR DISPONIBILIDAD DE GPU
# ============================================================

# Nota: Recordar que se debe de cambiar la configuración el tipo de ambiente en
# Colab, y en caso de hacerlo es necesario volver a correr el paso de
# importación de bibliotecas porque se cae el kernel.

# PyTorch es utilizado por muchas librerías modernas de Deep Learning.
# Aquí lo usamos solamente para verificar si Colab detecta GPU.

import torch

print("GPU Disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó GPU. El notebook puede ejecutarse en CPU, pero algunos modelos tardarán más.")

# ============================================================
# SECCIÓN 6 - TF-IDF
# ============================================================

# ¿Qué es TF-IDF?

TF-IDF significa: **Term Frequency - Inverse Document Frequency**.

En español se puede entender como: **Frecuencia del Término - Frecuencia Inversa del Documento**

Es una técnica clásica de Procesamiento del Lenguaje Natural utilizada para convertir texto en números.

Sin embargo, a diferencia de **Bag of Words**, **TF-IDF** no solo cuenta palabras. También intenta **medir qué tan importante** es una palabra dentro de un documento.

# Problema de Bag of Words

Bag of Words cuenta cuántas veces aparece cada palabra. Eso es útil, pero tiene un problema importante:

- Todas las palabras son tratadas de forma muy similar.

Por ejemplo, palabras como:

```text
- la
- el
- de
- es
- y
```

Pueden aparecer muchas veces, pero normalmente no aportan tanto significado.

En cambio, palabras como:

```text
- transformers
- embeddings
- medicina
- fútbol
- pizza
```

pueden decir mucho más sobre el tema del documento.

# Intuición de TF-IDF

TF-IDF intenta responder esta pregunta:

# ¿Qué tan importante es una palabra dentro de un documento?

Para eso combina dos ideas:

| Componente | Pregunta que responde |
|---|---|
| TF | ¿Qué tan frecuente aparece esta palabra en este documento? |
| IDF | ¿Qué tan rara es esta palabra en todos los documentos? |

Una palabra será importante si:

- aparece en un documento,
- pero no aparece en todos los documentos.

Si una palabra aparece en todos los documentos, probablemente no ayuda mucho a distinguirlos.

# Ejemplo Conceptual

Supongamos estos documentos:

| Documento | Texto |
|---|---|
| 1 | Me gustan los perros |
| 2 | Los perros son increíbles |
| 3 | Me gusta la pizza |
| 4 | Los transformers cambiaron el NLP moderno |

La palabra: **perros** aparece en documentos relacionados con animales.

La palabra: **pizza** aparece solamente en un documento.

La palabra: **los** puede aparecer en varios documentos, pero no necesariamente aporta mucho significado.

TF-IDF intenta asignar mayor **peso** a palabras útiles para diferenciar documentos.

In [ ]:
# ============================================================
# DATASET PEQUEÑO PARA ENTENDER TF-IDF
# ============================================================

# Cada elemento de esta lista representa un documento.
# En este caso usamos frases cortas para que sea fácil interpretar los resultados.

documentos = [
    "Me gustan los perros",
    "Los perros son increíbles",
    "Me gusta la pizza",
    "Los transformers cambiaron el NLP moderno"
]

# Mostrar los documentos numerados.
for indice, documento in enumerate(documentos):
    print(f"Documento {indice}: {documento}")

In [ ]:
# ============================================================
# CREAR EL VECTORIZADOR TF-IDF
# ============================================================

# TfidfVectorizer convierte texto en una matriz numérica.
# Cada fila representa un documento.
# Cada columna representa una palabra del vocabulario.
# Cada valor representa la importancia TF-IDF de esa palabra.

tfidf = TfidfVectorizer()

In [ ]:
# ============================================================
# APLICAR A LOS DOCUMENTOS
# ============================================================

# fit_transform realiza dos pasos:
#
# fit:
# aprende el vocabulario y las estadísticas del conjunto de documentos.
#
# transform:
# convierte los documentos en valores numéricos TF-IDF.

X_tfidf = tfidf.fit_transform(documentos)

In [ ]:
# ============================================================
# MOSTRAR DIMENSIONES DE LA MATRIZ
# ============================================================

# Mostrar dimensiones de la matriz.
# Filas = documentos
# Columnas = palabras únicas del vocabulario

print("Dimensiones de la matriz TF-IDF:", X_tfidf.shape)

In [ ]:
# ============================================================
# MOSTAR COMO MATRIZ NUMERICA
# ============================================================

print(X_tfidf.toarray())

In [ ]:
# ============================================================
# VISUALIZAR EL VOCABULARIO
# ============================================================

# get_feature_names_out devuelve las palabras que el vectorizador detectó.

vocabulario = tfidf.get_feature_names_out()

print("Vocabulario detectado:")
print(vocabulario)

In [ ]:
# ============================================================
# CONVERTIR TF-IDF A DATAFRAME
# ============================================================

# La matriz generada por TF-IDF es una matriz dispersa.
# Para visualizarla mejor, la convertimos a un DataFrame de Pandas.

df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vocabulario
)

df_tfidf

# ¿Cómo interpretar la tabla?

- Cada fila representa un documento.

- Cada columna representa una palabra.

- Cada valor representa la importancia TF-IDF de esa palabra en ese documento.

- Un valor de 0 significa que la palabra no aparece en ese documento.

- Un valor alto indica que esa palabra tiene más importancia dentro del documento.

In [ ]:
# ============================================================
# HEATMAP DE TF-IDF
# ============================================================

# Un heatmap permite visualizar mejor qué palabras tienen mayor peso.
# Los valores más altos se observan con mayor intensidad.

plt.figure(figsize=(12, 5))

sns.heatmap(
    df_tfidf,
    annot=True,
    cmap="Blues"
)

plt.title("Matriz TF-IDF")
plt.xlabel("Palabras")
plt.ylabel("Documentos")

plt.show()

In [ ]:
# ============================================================
# PALABRAS CON MAYOR IMPORTANCIA TOTAL
# ============================================================

# Sumamos los valores TF-IDF de cada palabra en todos los documentos.
# Esto nos da una idea de cuáles palabras tienen mayor importancia general.

importancia_total = df_tfidf.sum().sort_values(ascending=False)

importancia_total

In [ ]:
# ============================================================
# GRÁFICO DE PALABRAS MÁS IMPORTANTES
# ============================================================

# Graficamos las palabras con mayor importancia acumulada.

importancia_total.head(10).plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Palabras con Mayor Importancia TF-IDF")
plt.xlabel("Palabras")
plt.ylabel("Importancia Total")

plt.show()

# Actividad de Análisis

Responda:

1. ¿Qué palabras obtuvieron mayor peso?
2. ¿Por qué algunas palabras tienen valor 0 en ciertos documentos?
3. ¿Qué diferencia observa entre TF-IDF y Bag of Words?
4. ¿TF-IDF entiende realmente el significado de las palabras?
5. ¿Qué cree que pasaría si agregamos más documentos?

# ============================================================
# SECCIÓN 7 - Ejercicio de TF-IDF con un Libro en TXT
# ============================================================

# Objetivo

En esta sección se aplicará TF-IDF sobre un texto real.

La idea es descargar un libro en formato TXT, cargarlo en Google Colab, dividirlo en fragmentos y analizar qué palabras aparecen como más relevantes.

Este ejercicio permite comprender cómo TF-IDF se comporta con textos más grandes.

# Parte 1: Descargar un Libro en Formato TXT

Ingrese a Project Gutenberg:

https://www.gutenberg.org/

Seleccione un libro de su preferencia y descargue la versión:

```text
Plain Text UTF-8
```

Libros recomendados:

- Frankenstein
- Dracula
- Sherlock Holmes
- Pride and Prejudice
- Alice in Wonderland
- Moby Dick

También puede usar cualquier otro libro disponible en formato TXT.

# Parte 2: Subir el Archivo a Google Colab

Para subir el archivo:

1. Abra el panel izquierdo de Colab.
2. Seleccione el icono de archivos.
3. Presione el botón "Subir".
4. Seleccione el archivo `.txt` descargado.
5. Copie el nombre exacto del archivo.

En la siguiente celda deberá reemplazar:

```python
"libro.txt"
```

por el nombre real del archivo.

In [ ]:
# ============================================================
# OPCIÓN A: CARGAR UN LIBRO SUBIDO MANUALMENTE
# ============================================================

# Cambie este nombre por el nombre exacto de su archivo.
# Ejemplo:
# nombre_archivo = "pg84.txt"

nombre_archivo = "libro.txt"

# Esta celda intenta abrir el archivo indicado.
# Si todavía no ha subido el archivo, mostrará un mensaje de error controlado.

try:
    with open(nombre_archivo, "r", encoding="utf-8") as file:
        texto_libro = file.read()

    print("Archivo cargado correctamente.")
    print("Cantidad de caracteres:", len(texto_libro))

except FileNotFoundError:
    print("No se encontró el archivo.")
    print("Verifique que el archivo fue subido a Colab y que el nombre es correcto.")

# Opción Alternativa

Si no desea subir un archivo manualmente, puede descargar un texto directamente desde una URL.

El siguiente ejemplo descarga Frankenstein desde Project Gutenberg.

Esta opción ayuda a que todos puedan ejecutar el ejercicio aunque no hayan descargado un archivo manualmente.

In [ ]:
# ============================================================
# OPCIÓN B: DESCARGAR UN LIBRO DESDE UNA URL
# ============================================================

# Esta celda descarga un libro en formato texto usando requests.
# Se utiliza una URL pública de Project Gutenberg.

import requests

url = "https://www.gutenberg.org/cache/epub/84/pg84.txt"

respuesta = requests.get(url)

if respuesta.status_code == 200:
    texto_libro = respuesta.text
    print("Libro descargado correctamente desde Project Gutenberg.")
    print("Cantidad de caracteres:", len(texto_libro))
else:
    print("No se pudo descargar el libro.")
    print("Código de respuesta:", respuesta.status_code)

In [ ]:
# ============================================================
# VISUALIZAR UNA PARTE DEL TEXTO
# ============================================================

# Mostramos los primeros 2000 caracteres.
# Esto permite verificar que el texto se cargó correctamente.

print(texto_libro[:2000])

# Parte 3: Dividir el Libro en Fragmentos

TF-IDF trabaja comparando documentos.

Un libro completo puede ser considerado un documento muy grande, pero eso no permite comparar partes internas.

Por eso vamos a dividir el libro en fragmentos más pequeños.

En este caso usaremos puntos como separadores iniciales.

Esto no es perfecto, pero sirve para comenzar.

In [ ]:
# ============================================================
# DIVIDIR TEXTO EN FRAGMENTOS
# ============================================================

# split(".") divide el texto cada vez que encuentra un punto.
# Cada fragmento será tratado como un documento.

fragmentos = texto_libro.split(".")

# Limpiar fragmentos:
# - strip elimina espacios al inicio y al final.
# - se eliminan fragmentos demasiado cortos.

fragmentos = [
    fragmento.strip()
    for fragmento in fragmentos
    if len(fragmento.strip()) > 100
]

print("Cantidad de fragmentos:", len(fragmentos))

In [ ]:
# ============================================================
# MOSTRAR ALGUNOS FRAGMENTOS
# ============================================================

# Mostramos los primeros 5 fragmentos para verificar el resultado.

for i in range(5):
    print(f"Fragmento {i}:")
    print(fragmentos[i])
    print()

# Parte 4: Aplicar TF-IDF al Libro

Ahora aplicaremos TF-IDF a los fragmentos.

Usaremos `stop_words="english"` porque muchos libros de Project Gutenberg están en inglés.

Las stop words son palabras muy comunes como:

```text
the
and
is
of
```

Estas palabras normalmente no aportan mucho significado.

In [ ]:
# ============================================================
# TF-IDF SOBRE FRAGMENTOS DEL LIBRO
# ============================================================

# max_features limita la cantidad de palabras del vocabulario.
# Esto evita crear una matriz demasiado grande.
#
# stop_words="english" elimina palabras comunes en inglés.

tfidf_libro = TfidfVectorizer(
    stop_words="english",
    max_features=1000
)

X_libro = tfidf_libro.fit_transform(fragmentos)

print("Dimensiones de la matriz TF-IDF del libro:", X_libro.shape)

In [ ]:
# ============================================================
# CREAR DATAFRAME TF-IDF DEL LIBRO
# ============================================================

vocabulario_libro = tfidf_libro.get_feature_names_out()

df_libro_tfidf = pd.DataFrame(
    X_libro.toarray(),
    columns=vocabulario_libro
)

df_libro_tfidf.head()

In [ ]:
# ============================================================
# PALABRAS MÁS IMPORTANTES DEL LIBRO
# ============================================================

# Sumamos la importancia TF-IDF de cada palabra en todos los fragmentos.

importancia_libro = df_libro_tfidf.sum().sort_values(ascending=False)

importancia_libro.head(20)

In [ ]:
# ============================================================
# GRAFICAR PALABRAS MÁS IMPORTANTES
# ============================================================

plt.figure(figsize=(12, 5))

importancia_libro.head(20).plot(kind="bar")

plt.title("Palabras Más Importantes del Libro según TF-IDF")
plt.xlabel("Palabras")
plt.ylabel("Importancia TF-IDF Total")

plt.show()

# Parte 5: Buscar Fragmentos Relevantes

Ahora vamos a utilizar TF-IDF para buscar fragmentos relevantes a partir de una consulta.

Esto es parecido a un buscador simple.

Importante:

Este buscador todavía no entiende significado profundo. Principalmente busca coincidencias de palabras y pesos estadísticos.

In [ ]:
# ============================================================
# CONSULTA DEL USUARIO
# ============================================================

# Puede modificar esta consulta.
# Use palabras relacionadas con el libro.

consulta = "monster creation science"

print("Consulta:", consulta)

In [ ]:
# ============================================================
# TRANSFORMAR CONSULTA A TF-IDF
# ============================================================

# Usamos el mismo vectorizador que fue entrenado con los fragmentos.
# No usamos fit_transform aquí.
#
# Usamos transform porque el vocabulario ya fue aprendido.

consulta_tfidf = tfidf_libro.transform([consulta])

In [ ]:
# ============================================================
# CALCULAR SIMILITUD ENTRE CONSULTA Y FRAGMENTOS
# ============================================================

# cosine_similarity compara el vector de la consulta
# contra todos los fragmentos del libro.

scores = cosine_similarity(
    consulta_tfidf,
    X_libro
).flatten()

print("Cantidad de scores:", len(scores))

In [ ]:
# ============================================================
# MOSTRAR LOS FRAGMENTOS MÁS RELEVANTES
# ============================================================

# argsort ordena los índices de menor a mayor.
# Usamos [-5:] para tomar los 5 más altos.
# Luego invertimos con [::-1] para mostrarlos de mayor a menor.

top_indices = scores.argsort()[-5:][::-1]

for rank, indice in enumerate(top_indices, start=1):
    print(f"Resultado {rank}")
    print("Score:", scores[indice])
    print("Fragmento:")
    print(fragmentos[indice][:1000])
    print()

# Actividad de Experimentación

Modifique la consulta y observe los resultados.

Pruebe consultas como:

```text
love and family
fear and darkness
science and discovery
death and life
```

Responda:

1. ¿Los fragmentos recuperados tienen sentido?
2. ¿Qué pasa si usa sinónimos?
3. ¿Qué pasa si usa palabras que no están en el libro?
4. ¿Qué limitación observa en este buscador?

# ============================================================
# SECCIÓN 8 - Similitud entre Fragmentos
# ============================================================

TF-IDF también puede utilizarse para comparar documentos entre sí.

En este caso, compararemos fragmentos del libro para identificar cuáles son más parecidos.

In [ ]:
# ============================================================
# CALCULAR SIMILITUD ENTRE FRAGMENTOS
# ============================================================

# Para no saturar memoria ni visualización, tomamos solo los primeros 30 fragmentos.

cantidad_fragmentos = 30

X_muestra = X_libro[:cantidad_fragmentos]

similaridad_fragmentos = cosine_similarity(X_muestra)

print("Dimensiones de la matriz de similitud:", similaridad_fragmentos.shape)

In [ ]:
# ============================================================
# VISUALIZAR SIMILITUD ENTRE FRAGMENTOS
# ============================================================

plt.figure(figsize=(10, 8))

sns.heatmap(
    similaridad_fragmentos,
    cmap="viridis"
)

plt.title("Similitud entre Fragmentos usando TF-IDF")
plt.xlabel("Fragmento")
plt.ylabel("Fragmento")

plt.show()

In [ ]:
# ============================================================
# BUSCAR FRAGMENTOS SIMILARES A UNO SELECCIONADO
# ============================================================

# Seleccione un índice de fragmento.
# Puede cambiar este valor.

indice_base = 0

scores_fragmento = cosine_similarity(
    X_libro[indice_base],
    X_libro
).flatten()

indices_similares = scores_fragmento.argsort()[-6:][::-1]

print("Fragmento base:")
print(fragmentos[indice_base][:1000])
print()

print("Fragmentos más similares:")

for indice in indices_similares:
    if indice == indice_base:
        continue

    print("Índice:", indice)
    print("Score:", scores_fragmento[indice])
    print(fragmentos[indice][:700])
    print()

# Reflexión sobre Similitud

Responda:

1. ¿Los fragmentos similares realmente hablan de temas parecidos?
2. ¿La similitud parece depender de palabras exactas?
3. ¿Qué pasaría si dos fragmentos hablan del mismo tema con palabras distintas?
4. ¿Qué técnica moderna podría resolver mejor ese problema?

# ¿Qué hemos hecho hasta ahora?

En las secciones anteriores aprendimos a convertir texto en números utilizando:

- Bag of Words
- TF-IDF

Por ejemplo:

Texto:

```text
Los transformers revolucionaron NLP
```

TF-IDF:

```text
[0.21, 0.45, 0.00, 0.62, ...]
```

Ahora tenemos una representación numérica.

---

# La Gran Pregunta

Si ya podemos convertir texto en números:

# ¿Podemos entrenar una computadora para reconocer automáticamente de qué trata un texto?

Por ejemplo:

```text
Los transformers revolucionaron NLP
```

Tecnología

---

```text
El delantero anotó tres goles
```

Deportes

---

```text
La red neuronal mejoró la precisión
```

Tecnología

---

```text
El equipo ganó el campeonato
```

Deportes

---

Eso es exactamente lo que haremos en esta sección.

Construiremos nuestro primer modelo de Machine Learning para clasificación de texto.

# ¿Qué significa Clasificación?

Clasificar significa asignar una categoría a un elemento.

Ejemplos:

| Entrada | Categoría |
|----------|----------|
| Correo electrónico | Spam o No Spam |
| Comentario | Positivo o Negativo |
| Noticia | Deportes o Tecnología |
| Ticket de soporte | Hardware o Software |

---

# ============================================================
# SECCIÓN 9 - Clasificación de Texto con Machine Learning Tradicional
# ============================================================

Ahora construiremos un modelo clásico de clasificación de texto.

Este ejemplo muestra cómo TF-IDF puede usarse como entrada para un modelo de Machine Learning.

Usaremos un dataset pequeño creado manualmente para clasificar mensajes en dos categorías:

- tecnología
- deportes



In [ ]:
# ============================================================
# CREAR DATASET DE CLASIFICACIÓN
# ============================================================

# Cada texto tiene una etiqueta.
# 1 representa tecnología.
# 0 representa deportes.

textos = [
    "Los transformers revolucionaron el procesamiento del lenguaje natural",
    "El modelo de inteligencia artificial fue entrenado con muchos datos",
    "La red neuronal mejoró su precisión después del entrenamiento",
    "Los embeddings permiten representar significado semántico",
    "El equipo ganó el partido en el último minuto",
    "El jugador anotó tres goles durante la final",
    "La selección nacional entrenó antes del campeonato",
    "El estadio estaba lleno durante el partido",
    "La búsqueda semántica utiliza vectores y similitud",
    "El algoritmo clasificó correctamente los documentos",
    "El entrenador cambió la estrategia del equipo",
    "El torneo internacional comenzó esta semana"
]

etiquetas = [
    1, 1, 1, 1,
    0, 0, 0, 0,
    1, 1,
    0, 0
]

df_clasificacion = pd.DataFrame({
    "texto": textos,
    "etiqueta": etiquetas
})

df_clasificacion

In [ ]:
# ============================================================
# MAPEAR ETIQUETAS A TEXTO
# ============================================================

# Esto ayuda a interpretar mejor los resultados.

df_clasificacion["categoria"] = df_clasificacion["etiqueta"].map({
    1: "tecnología",
    0: "deportes"
})

df_clasificacion

In [ ]:
# ============================================================
# DIVIDIR DATOS EN ENTRENAMIENTO Y PRUEBA
# ============================================================

# train_test_split separa datos para entrenar y evaluar.
#
# test_size=0.25 significa que 25% será usado para prueba.
# random_state permite reproducibilidad.

X_train, X_test, y_train, y_test = train_test_split(
    df_clasificacion["texto"],
    df_clasificacion["etiqueta"],
    test_size=0.25,
    random_state=42,
    stratify=df_clasificacion["etiqueta"]
)

print("Cantidad de ejemplos de entrenamiento:", len(X_train))
print("Cantidad de ejemplos de prueba:", len(X_test))

In [ ]:
# ============================================================
# VECTORIZAR TEXTO CON TF-IDF
# ============================================================

# Creamos un nuevo vectorizador para este problema.

vectorizador_clf = TfidfVectorizer()

# fit_transform solo se aplica al conjunto de entrenamiento.
# Esto evita que el modelo vea información del conjunto de prueba.

X_train_tfidf = vectorizador_clf.fit_transform(X_train)

# transform se aplica al conjunto de prueba usando el vocabulario aprendido.
X_test_tfidf = vectorizador_clf.transform(X_test)

print("Dimensiones entrenamiento:", X_train_tfidf.shape)
print("Dimensiones prueba:", X_test_tfidf.shape)

In [ ]:
# ============================================================
# ENTRENAR MODELO NAIVE BAYES
# ============================================================

# MultinomialNB es un modelo clásico muy usado en clasificación de texto.
# Funciona bien con conteos de palabras y representaciones TF-IDF.

modelo_nb = MultinomialNB()

modelo_nb.fit(X_train_tfidf, y_train)

In [ ]:
# ============================================================
# REALIZAR PREDICCIONES
# ============================================================

predicciones_nb = modelo_nb.predict(X_test_tfidf)

predicciones_nb

In [ ]:
# ============================================================
# EVALUAR MODELO
# ============================================================

accuracy = accuracy_score(y_test, predicciones_nb)

print("Accuracy:", accuracy)
print()

print("Reporte de clasificación:")
print(classification_report(
    y_test,
    predicciones_nb,
    target_names=["deportes", "tecnología"]
))

In [ ]:
# ============================================================
# MATRIZ DE CONFUSIÓN
# ============================================================

cm = confusion_matrix(y_test, predicciones_nb)

plt.figure(figsize=(5, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["deportes", "tecnología"],
    yticklabels=["deportes", "tecnología"]
)

plt.title("Matriz de Confusión")
plt.xlabel("Predicción")
plt.ylabel("Valor Real")

plt.show()

# Probar Nuevos Textos

Ahora probaremos el modelo con frases nuevas.

Esto permite ver cómo el modelo generaliza a ejemplos que no vio durante entrenamiento.

In [ ]:
# ============================================================
# PROBAR MODELO CON NUEVOS TEXTOS
# ============================================================

nuevos_textos = [
    "El algoritmo utiliza embeddings para buscar documentos",
    "El delantero anotó un gol en la final",
    "La inteligencia artificial genera respuestas automáticas",
    "El campeonato terminó con empate"
]

nuevos_textos_tfidf = vectorizador_clf.transform(nuevos_textos)

nuevas_predicciones = modelo_nb.predict(nuevos_textos_tfidf)

for texto, prediccion in zip(nuevos_textos, nuevas_predicciones):
    categoria = "tecnología" if prediccion == 1 else "deportes"
    print("Texto:", texto)
    print("Predicción:", categoria)
    print()

# Reflexión

Responda:

1. ¿El modelo clasificó correctamente los textos nuevos?
2. ¿Qué palabras cree que influyeron más en la decisión?
3. ¿Qué pasaría si usamos textos ambiguos?
4. ¿Este modelo entiende realmente el significado?
5. ¿Qué limitaciones tendría en producción?

# ============================================================
# SECCIÓN 10 - Embeddings Semánticos
# ============================================================

TF-IDF es útil, pero tiene una limitación muy importante:

# No entiende significado semántico.

Por ejemplo:

```text
carro
automóvil
vehículo
```

son palabras relacionadas para un humano, pero TF-IDF las trata como palabras distintas.

Los embeddings modernos intentan resolver este problema.

Un embedding convierte texto en un vector numérico que representa significado.

In [ ]:
# ============================================================
# CARGAR MODELO DE EMBEDDINGS
# ============================================================

# all-MiniLM-L6-v2 es un modelo liviano y rápido.
# Es adecuado para prácticas en Colab.
#
# La primera vez puede tardar porque descarga el modelo.

modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# ============================================================
# FRASES DE EJEMPLO
# ============================================================

frases_embeddings = [
    "Me gustan los perros",
    "Los perros son animales increíbles",
    "Los gatos son mascotas populares",
    "La pizza está deliciosa",
    "Los transformers revolucionaron el NLP",
    "Los modelos de lenguaje usan embeddings"
]

for i, frase in enumerate(frases_embeddings):
    print(f"{i}: {frase}")

In [ ]:
# ============================================================
# GENERAR EMBEDDINGS
# ============================================================

# encode convierte cada frase en un vector numérico.

embeddings = modelo_embeddings.encode(frases_embeddings)

print("Forma de los embeddings:", embeddings.shape)

# Interpretación

Si el resultado es:

```text
(6, 384)
```

significa:

- 6 frases
- cada frase representada por un vector de 384 dimensiones

No necesitamos interpretar manualmente cada dimensión.

Lo importante es que frases con significado parecido tendrán vectores más similares.

In [ ]:
# ============================================================
# CALCULAR SIMILITUD COSENO ENTRE EMBEDDINGS
# ============================================================

matriz_similitud_embeddings = cosine_similarity(embeddings)

matriz_similitud_embeddings

In [ ]:
# ============================================================
# VISUALIZAR SIMILITUD SEMÁNTICA
# ============================================================

plt.figure(figsize=(10, 7))

sns.heatmap(
    matriz_similitud_embeddings,
    annot=True,
    cmap="Greens",
    xticklabels=frases_embeddings,
    yticklabels=frases_embeddings
)

plt.title("Similitud Semántica entre Frases usando Embeddings")

plt.show()

# Comparación Conceptual

| Técnica | Qué compara principalmente |
|---|---|
| Bag of Words | conteo de palabras |
| TF-IDF | importancia estadística de palabras |
| Embeddings | significado semántico |

Los embeddings permiten que dos frases sean similares aunque no usen exactamente las mismas palabras.

# ============================================================
# SECCIÓN 11 - Buscador Semántico con Embeddings
# ============================================================

Ahora construiremos un pequeño buscador semántico.

La idea es:

1. crear una colección de documentos,
2. convertirlos a embeddings,
3. convertir una consulta a embedding,
4. calcular similitud,
5. devolver el documento más relacionado.

Esto será la base conceptual para RAG más adelante.

In [ ]:
# ============================================================
# DOCUMENTOS PARA BÚSQUEDA SEMÁNTICA
# ============================================================

documentos_busqueda = [
    "La inteligencia artificial está transformando la medicina moderna.",
    "Los modelos de lenguaje pueden responder preguntas y resumir texto.",
    "El fútbol es uno de los deportes más populares del mundo.",
    "La pizza italiana es famosa por su masa y sus ingredientes.",
    "Los embeddings permiten representar significado en forma de vectores.",
    "Los sistemas RAG combinan búsqueda de información con generación de respuestas.",
    "El aprendizaje profundo utiliza redes neuronales con muchas capas.",
    "La privacidad de los datos es crítica en sistemas de inteligencia artificial."
]

for i, doc in enumerate(documentos_busqueda):
    print(f"{i}: {doc}")

In [ ]:
# ============================================================
# GENERAR EMBEDDINGS DE DOCUMENTOS
# ============================================================

embeddings_documentos = modelo_embeddings.encode(documentos_busqueda)

print("Forma:", embeddings_documentos.shape)

In [ ]:
# ============================================================
# CREAR CONSULTA
# ============================================================

# Puede modificar esta consulta y observar cómo cambian los resultados.

consulta = "¿Cómo ayudan los vectores a buscar información?"

embedding_consulta = modelo_embeddings.encode([consulta])

print("Consulta:", consulta)

In [ ]:
# ============================================================
# CALCULAR SIMILITUD ENTRE CONSULTA Y DOCUMENTOS
# ============================================================

scores_semanticos = cosine_similarity(
    embedding_consulta,
    embeddings_documentos
).flatten()

scores_semanticos

In [ ]:
# ============================================================
# ORDENAR RESULTADOS POR SIMILITUD
# ============================================================

indices_ordenados = scores_semanticos.argsort()[::-1]

for rank, indice in enumerate(indices_ordenados[:5], start=1):
    print(f"Resultado {rank}")
    print("Score:", scores_semanticos[indice])
    print("Documento:", documentos_busqueda[indice])
    print()

# Actividad

Pruebe las siguientes consultas:

```text
¿Cómo funciona RAG?
¿Qué problemas existen con privacidad?
¿Qué es aprendizaje profundo?
¿Cuál documento habla de comida?
```

Luego responda:

1. ¿Los resultados tienen sentido?
2. ¿El sistema recupera documentos por significado o por palabras exactas?
3. ¿Qué diferencia observa respecto a TF-IDF?
4. ¿Por qué esto es importante para asistentes inteligentes?

# ============================================================
# SECCIÓN 12 - Comparación entre TF-IDF y Embeddings
# ============================================================

En esta sección compararemos ambos enfoques.

Queremos observar qué ocurre cuando dos textos tienen significado parecido, pero palabras distintas.

In [ ]:
# ============================================================
# EJEMPLO DE SINÓNIMOS Y SIGNIFICADO
# ============================================================

frases_comparacion = [
    "El carro es rápido",
    "El automóvil se mueve velozmente",
    "La pizza tiene queso",
    "El vehículo alcanza alta velocidad"
]

for i, frase in enumerate(frases_comparacion):
    print(f"{i}: {frase}")

In [ ]:
# ============================================================
# SIMILITUD CON TF-IDF
# ============================================================

tfidf_comparacion = TfidfVectorizer()

X_comparacion_tfidf = tfidf_comparacion.fit_transform(frases_comparacion)

sim_tfidf = cosine_similarity(X_comparacion_tfidf)

plt.figure(figsize=(7, 5))

sns.heatmap(
    sim_tfidf,
    annot=True,
    cmap="Blues",
    xticklabels=frases_comparacion,
    yticklabels=frases_comparacion
)

plt.title("Similitud usando TF-IDF")

plt.show()

In [ ]:
# ============================================================
# SIMILITUD CON EMBEDDINGS
# ============================================================

embeddings_comparacion = modelo_embeddings.encode(frases_comparacion)

sim_embeddings = cosine_similarity(embeddings_comparacion)

plt.figure(figsize=(7, 5))

sns.heatmap(
    sim_embeddings,
    annot=True,
    cmap="Greens",
    xticklabels=frases_comparacion,
    yticklabels=frases_comparacion
)

plt.title("Similitud usando Embeddings")

plt.show()

# Análisis

Compare ambos heatmaps.

Preguntas:

1. ¿Cuál técnica detecta mejor la relación entre carro, automóvil y vehículo?
2. ¿Cuál depende más de palabras exactas?
3. ¿Cuál parece capturar mejor significado?
4. ¿Por qué TF-IDF sigue siendo útil si embeddings son más modernos?

# ============================================================
# SECCIÓN 13 - Introducción Práctica a Transformers
# ============================================================

Los transformers revolucionaron el Procesamiento del Lenguaje Natural.

A diferencia de enfoques tradicionales, los transformers pueden capturar relaciones contextuales más complejas.

Modelos modernos como:

- BERT
- GPT
- T5
- LLaMA
- Gemini
- Claude

están basados en arquitecturas transformer o derivadas de ellas.

En esta sección utilizaremos modelos preentrenados mediante HuggingFace.

# ¿Qué es un Pipeline de HuggingFace?

Un pipeline es una forma sencilla de usar modelos preentrenados.

Permite ejecutar tareas como:

- análisis de sentimientos
- resumen de texto
- question answering
- generación de texto
- traducción
- clasificación

sin tener que entrenar un modelo desde cero.

In [ ]:
# ============================================================
# PIPELINE DE ANÁLISIS DE SENTIMIENTOS
# ============================================================

# Este pipeline carga un modelo preentrenado para clasificar sentimiento.
# Puede tardar la primera vez porque descarga el modelo.

analizador_sentimiento = pipeline("sentiment-analysis")

In [ ]:
# ============================================================
# PROBAR ANÁLISIS DE SENTIMIENTOS
# ============================================================

frases_sentimiento = [
    "I love this artificial intelligence course.",
    "This system is terrible and frustrating.",
    "The class was okay, nothing special.",
    "Excellent, the server crashed again."
]

resultados_sentimiento = analizador_sentimiento(frases_sentimiento)

for frase, resultado in zip(frases_sentimiento, resultados_sentimiento):
    print("Frase:", frase)
    print("Resultado:", resultado)
    print()

# Reflexión sobre Sentimiento

Observe especialmente esta frase:

```text
Excellent, the server crashed again.
```

Tiene una palabra positiva, pero el significado real puede ser negativo por sarcasmo.

Preguntas:

1. ¿El modelo detectó correctamente el sarcasmo?
2. ¿Qué limitaciones observa?
3. ¿Por qué el contexto es tan importante en NLP?

In [ ]:
# ============================================================
# PIPELINE DE QUESTION ANSWERING
# ============================================================

# Este pipeline responde preguntas usando un contexto dado.
# No inventa una respuesta desde conocimiento general.
# Busca la respuesta dentro del texto proporcionado.

qa = pipeline("question-answering")

In [ ]:
# ============================================================
# EJEMPLO DE QUESTION ANSWERING
# ============================================================

contexto = '''
Los transformers fueron introducidos en 2017 mediante el artículo
Attention Is All You Need. Esta arquitectura permitió mejorar muchas
tareas de procesamiento del lenguaje natural, incluyendo traducción,
resumen, clasificación y generación de texto.
'''

pregunta = "¿En qué año fueron introducidos los transformers?"

respuesta = qa(
    question=pregunta,
    context=contexto
)

respuesta

In [ ]:
# ============================================================
# PROBAR OTRA PREGUNTA
# ============================================================

pregunta = "¿Qué tareas mejoraron los transformers?"

respuesta = qa(
    question=pregunta,
    context=contexto
)

respuesta

# Actividad

Cambie el contexto y escriba sus propias preguntas.

Observe:

1. ¿El modelo responde solo con información del contexto?
2. ¿Qué pasa si la respuesta no aparece en el contexto?
3. ¿El modelo admite que no sabe?
4. ¿Qué riesgos tendría esto en un sistema real?

# ============================================================
# SECCIÓN 14 - Resumen Automático
# ============================================================

El resumen automático es una tarea clásica de NLP.

Consiste en tomar un texto largo y producir una versión más corta.

Esta tarea es muy útil en:

- análisis documental
- sistemas legales
- investigación académica
- soporte al cliente
- análisis de noticias

In [ ]:
# ============================================================
# PIPELINE DE RESUMEN
# ============================================================

# El modelo por defecto puede tardar en descargarse.
# Si Colab muestra advertencias, normalmente se pueden ignorar.

resumidor = pipeline("summarization")

In [ ]:
# ============================================================
# TEXTO PARA RESUMIR
# ============================================================

texto_largo = '''
Artificial intelligence is transforming modern industries by enabling
organizations to automate complex tasks, analyze large volumes of data,
and improve decision-making processes. In healthcare, AI systems are used
to support diagnosis, identify patterns in medical images, and assist in
patient management. In finance, artificial intelligence helps detect fraud,
assess risk, and personalize services. However, these technologies also
introduce challenges related to privacy, bias, transparency, and security.
For this reason, organizations must evaluate AI systems carefully before
deploying them in real-world environments.
'''

resumen = resumidor(
    texto_largo,
    max_length=80,
    min_length=25,
    do_sample=False
)

resumen

# Reflexión sobre Resumen

Responda:

1. ¿El resumen conserva las ideas principales?
2. ¿El modelo omitió algo importante?
3. ¿El resumen agregó información que no estaba en el texto?
4. ¿En qué contexto empresarial sería útil esta herramienta?

# ============================================================
# SECCIÓN 15 - Limitaciones, Errores y Alucinaciones
# ============================================================

Los modelos modernos son poderosos, pero no son perfectos.

Pueden cometer errores como:

- interpretar mal el contexto
- fallar con sarcasmo
- responder con exceso de confianza
- inventar información
- reproducir sesgos
- fallar con datos fuera de distribución

A esos errores donde el modelo inventa información se les suele llamar alucinaciones.

# Actividad de Análisis

Discuta los siguientes casos:

| Caso | Riesgo |
|---|---|
| Un chatbot médico inventa una recomendación |
| Un asistente legal resume mal una cláusula |
| Un sistema de soporte da instrucciones incorrectas |
| Un buscador semántico recupera documentos irrelevantes |
| Un modelo clasifica mal mensajes sarcásticos |

Preguntas:

1. ¿Cuál sería el impacto?
2. ¿Cómo se podría reducir el riesgo?
3. ¿Se requiere supervisión humana?
4. ¿Qué métricas o pruebas serían necesarias?

# ============================================================
# SECCIÓN 16 - Mini Proyecto Integrador de Clase
# ============================================================

# Objetivo

Construir un pequeño sistema de búsqueda y análisis de texto que permita comparar:

- búsqueda basada en TF-IDF
- búsqueda basada en embeddings

El estudiante deberá observar cuál enfoque funciona mejor según el tipo de consulta.

In [ ]:
# ============================================================
# DOCUMENTOS DEL MINI PROYECTO
# ============================================================

corpus = [
    "Los transformers utilizan mecanismos de atención para procesar lenguaje.",
    "TF-IDF asigna peso a las palabras según su frecuencia e importancia.",
    "Los embeddings representan significado semántico en vectores.",
    "RAG combina recuperación de información con generación de texto.",
    "Los modelos de lenguaje pueden producir respuestas incorrectas.",
    "La privacidad es un aspecto crítico en sistemas de inteligencia artificial.",
    "El fútbol requiere estrategia, entrenamiento y trabajo en equipo.",
    "La pizza italiana tradicional se prepara con masa, tomate y queso."
]

consulta = "¿Cómo se representa el significado de un texto?"

print("Consulta:", consulta)

In [ ]:
# ============================================================
# BÚSQUEDA CON TF-IDF
# ============================================================

vectorizador_tfidf = TfidfVectorizer()

X_corpus_tfidf = vectorizador_tfidf.fit_transform(corpus)

consulta_tfidf = vectorizador_tfidf.transform([consulta])

scores_tfidf = cosine_similarity(
    consulta_tfidf,
    X_corpus_tfidf
).flatten()

indices_tfidf = scores_tfidf.argsort()[::-1]

print("Resultados usando TF-IDF:")
print()

for rank, indice in enumerate(indices_tfidf[:3], start=1):
    print(f"Resultado {rank}")
    print("Score:", scores_tfidf[indice])
    print(corpus[indice])
    print()

In [ ]:
# ============================================================
# BÚSQUEDA CON EMBEDDINGS
# ============================================================

embeddings_corpus = modelo_embeddings.encode(corpus)

embedding_consulta = modelo_embeddings.encode([consulta])

scores_embeddings = cosine_similarity(
    embedding_consulta,
    embeddings_corpus
).flatten()

indices_embeddings = scores_embeddings.argsort()[::-1]

print("Resultados usando Embeddings:")
print()

for rank, indice in enumerate(indices_embeddings[:3], start=1):
    print(f"Resultado {rank}")
    print("Score:", scores_embeddings[indice])
    print(corpus[indice])
    print()

# Análisis Final del Mini Proyecto

Responda:

1. ¿Cuál enfoque recuperó mejores resultados?
2. ¿TF-IDF funcionó bien cuando había palabras exactas?
3. ¿Embeddings funcionó mejor con significado?
4. ¿Cuál enfoque sería más adecuado para un asistente conversacional?
5. ¿Por qué RAG suele utilizar embeddings?

# ============================================================
# Cierre de la Clase 2
# ============================================================

En esta clase se trabajaron varios conceptos fundamentales para NLP moderno:

- TF-IDF
- análisis de importancia de palabras
- carga y análisis de libros en TXT
- similitud entre fragmentos
- clasificación de texto con Machine Learning tradicional
- embeddings semánticos
- búsqueda semántica
- transformers mediante HuggingFace
- análisis de errores y limitaciones

# Ideas Clave

TF-IDF sigue siendo útil porque es rápido, interpretable y sencillo.

Sin embargo, TF-IDF no comprende significado profundo.

Los embeddings permiten representar significado semántico y son una base fundamental para sistemas modernos como RAG.

Los transformers permiten resolver tareas complejas de NLP usando modelos preentrenados, pero deben evaluarse cuidadosamente.

# Retos Opcionales

1. Descargar otro libro y repetir el análisis TF-IDF.
2. Crear un dataset propio de clasificación.
3. Comparar Naive Bayes contra Regresión Logística.
4. Probar consultas ambiguas en búsqueda semántica.
5. Probar frases con sarcasmo en el pipeline de sentimientos.
6. Investigar qué es chunking y por qué es importante en RAG.